# 🌊 Data Processing & Exploratory Data Analysis (EDA)
Đồ án: **XÂY DỰNG NỀN TẢNG WEBGIS DỰ BÁO XÂM NHẬP MẶN VÀ ĐIỀU PHỐI VẬN HÀNH CỐNG THỦY LỢI TẠI ĐBSCL**

Trong notebook này, chúng ta sẽ thực hiện:
1. Tải Master Timeseries (đã gộp các nguồn dữ liệu).
2. Làm sạch dữ liệu (Xử lý Missing Values, Duplicates).
3. Trực quan hóa & Phân tích chuyên sâu (EDA).
4. Feature Engineering & Xuất dữ liệu cho Machine Learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Thiết lập font và style cho biểu đồ đẹp hơn
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

MASTER_CSV = r'd:/Study/TLCN/data/processed/master_timeseries.csv'
df = pd.read_csv(MASTER_CSV)
df['month_start'] = pd.to_datetime(df['month_start'])
df.head()

## 1. Data Cleaning (Làm sạch dữ liệu)
Xử lý các bản ghi bị trùng lặp và các giá trị NaN (Missing values) do quá trình merge Outer Sinh ra.

In [ ]:
# 1. Xóa cột bị trùng lặp (Duplicates)
duplicates_count = df.duplicated().sum()
print(f"Số dòng trùng lặp: {duplicates_count}")
df = df.drop_duplicates()

# 2. Xử lý Missing Values (NaN)
print("\nSố lượng Missing Values trên từng cột:")
print(df.isna().sum())

# Chiến lược Fill NaN: Dùng phép nội suy (Interpolation) tuyến tính cho dữ liệu chuỗi thời gian,
# và fillna(method='bfill') / 'ffill' cho những điểm không thể nội suy ở đầu chót ranh giới.
df = df.sort_values(by=['station', 'month_start'])
for col in df.columns:
    if df[col].dtype in ['float64', 'int64']:
        df[col] = df.groupby('station')[col].transform(lambda group: group.interpolate(method='linear', limit_direction='both'))

print("\nSau khi xử lý Missing Values:")
print(df.isna().sum())

Chuẩn bị DataFrame phân cách riêng cho **Tân Châu** (Lưu lượng) và **Mỹ Tho** (Độ mặn) để dễ Plot.

In [ ]:
df_tanchau = df[df['station'] == 'TanChau'].set_index('month_start')
df_mytho = df[df['station'] == 'MyTho'].set_index('month_start')

# Gộp 2 trạm lại cho các phân tích đối chiếu
df_merged = pd.merge(
    df_mytho[['conductivity_mS_per_m', 'temperature_mean', 'evapotranspiration_sum', 'precipitation_sum']],
    df_tanchau[['glofas_discharge_m3s']], 
    left_index=True, 
    right_index=True, 
    how='inner'
)
# Tính toán Độ Mặn (Salinity ppt) từ Khả năng dẫn điện (Conductivity mS/m)
df_merged['salinity_ppt'] = df_merged['conductivity_mS_per_m'] * 0.64


## 2. Trực quan hóa & Phân tích chuyên sâu (EDA)
### A. Biến động Độ mặn và Lưu lượng nước theo thời gian
Quy luật "nước lên, mặn xuống" thể hiện qua việc lưu lượng xả từ thượng đỉnh Tân Châu cao thì hạn mặn hạ lưu Mỹ Tho càng thấp.

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

color = 'tab:blue'
ax1.set_xlabel('Thời gian', fontsize=12)
ax1.set_ylabel('Lưu lượng Tân Châu (m³/s)', color=color, fontsize=12, fontweight='bold')
ax1.plot(df_merged.index, df_merged['glofas_discharge_m3s'], color=color, linewidth=2, label='Lưu lượng (Tân Châu)')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  
color = 'tab:red'
ax2.set_ylabel('Độ mặn Mỹ Tho (ppt)', color=color, fontsize=12, fontweight='bold')  
ax2.plot(df_merged.index, df_merged['salinity_ppt'], color=color, linewidth=2, linestyle='--', label='Độ mặn (Mỹ Tho)')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Đối chiếu Biến động Lưu lượng Tân Châu và Độ mặn Mỹ Tho (Time-Series)', fontsize=15, fontweight='bold')
fig.tight_layout()
plt.show()

### B. Phân bố Độ mặn theo tháng (Tính Mùa vụ - Seasonality)
Nhóm boxplot này chứng minh Mỹ Tho chịu ảnh hưởng mùa vụ rất rõ. Tháng 3, 4, 5 (Mùa khô) là đỉnh mặn. Tháng 8-11 (Mùa mưa lũ) mặn bằng 0.

In [ ]:
df_merged['month'] = df_merged.index.month

plt.figure(figsize=(12, 6))
sns.boxplot(x='month', y='salinity_ppt', data=df_merged, palette='coolwarm')
plt.title('Phân bố Độ mặn Mỹ Tho theo từng tháng trong năm (Tính mùa vụ)', fontsize=15, fontweight='bold')
plt.xlabel('Tháng', fontsize=12)
plt.ylabel('Độ mặn (ppt)', fontsize=12)
plt.show()

### C. Ma trận Tương quan (Correlation Heatmap)
Mức độ ảnh hưởng qua lại giữa các yếu tố: Mưa, Lưu lượng, Độ mặn, Bốc hơi, Nhiệt độ...

In [ ]:
corr_cols = ['precipitation_sum', 'glofas_discharge_m3s', 'temperature_mean', 'evapotranspiration_sum', 'salinity_ppt']
corr_matrix = df_merged[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu', vmin=-1, vmax=1, fmt=".2f", linewidths=.5)
plt.title('Ma trận Tương quan (Pearson) giữa các yếu tố Khí tượng, Thủy văn, Độ mặn', fontsize=14, fontweight='bold')
plt.show()

### D. Scatter Plot: Tương quan Lưu lượng - Độ mặn
Cho thấy khi lưu lượng dưới ngưỡng 10,000 m3/s, độ mặn bắt đầu vọt lên rất cao, xác nhận ngưỡng cạn an toàn.

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(x='glofas_discharge_m3s', y='salinity_ppt', hue='month', palette='viridis', data=df_merged, s=80)
plt.title('Mối quan hệ giữa Lưu lượng Tân Châu và Độ mặn Mỹ Tho', fontsize=15, fontweight='bold')
plt.xlabel('Lưu lượng Tân Châu (m³/s)', fontsize=12)
plt.ylabel('Độ mặn Mỹ Tho (ppt)', fontsize=12)
plt.axvline(x=10000, color='red', linestyle='--', linewidth=1.5, label='Ngưỡng lưu lượng cạn (10,000 m³/s)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. Feature Engineering cho Machine Learning

In [ ]:
# Lag Features: Lưu lượng Tân Châu (1 tháng trước và 2 tháng trước)
df_merged['TanChau_Discharge_lag_1'] = df_merged['glofas_discharge_m3s'].shift(1)
df_merged['TanChau_Discharge_lag_2'] = df_merged['glofas_discharge_m3s'].shift(2)

# Lag Features: Độ mặn 1 tháng trước
df_merged['MyTho_Salinity_lag_1'] = df_merged['salinity_ppt'].shift(1)

# Loại bỏ các tháng bị NaN do shift
ml_df = df_merged.dropna().reset_index()

print('Dataset Shape sau Feature Engineering:', ml_df.shape)
ml_df.head()

In [ ]:
PROCESSED_DIR = pd.io.common.Path(r'd:/Study/TLCN/data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ml_df.to_csv(PROCESSED_DIR / 'ml_monthly_features.csv', index=False)
print('Đã lưu dữ liệu: ml_monthly_features.csv. Sẵn sàng cho Model Train.')